In [1]:
source(here::here("data-cleaning", "00a-parameters.r"))


Parallelization: FALSE 


# Libraries


In [2]:
# Update the grouper
system("git submodule update --init --recursive")

# List required packages
required_packages <- c(
  "data.table", # Fast data manipulation
  "here", # Simplifies file path management
  "tictoc", # Timing code execution
  "stringr", # String manipulation
  "stringi", # Unicode string processing
  "lubridate", # Date-time handling
  "profvis", # Profiling R code
  "hash", # Hashing utility
  "future", # Parallel processing
  "future.apply", # Parallelized apply functions
  "knitr", # Dynamic report generation
  "htmlwidgets", # Interactive HTML widgets
  "parallelly", # Advanced parallel computing
  "stringdist", # String distance calculations
  "parallel", # Base parallel computing
  "reticulate", # Interface to Python
  "bigrquery", # BigQuery client
  "jsonlite", # JSON parsing
  "googleCloudStorageR", # Google Cloud Storage access
  "haven", # Read/write Stata, SPSS, SAS files
  "fst", # Fast serialization
  "httr", # HTTP requests
  "ggplot2", # Data visualization
  "rmarkdown", # Dynamic markdown documents
  "digest", # Create cryptographic hashes
  "base64enc", # Base64 encoding/decoding
  "arrow", # Apache Arrow for fast data storage
  "tidyverse", # Collection of data science packages,
  "fasttime", # for fastPOSIXct
  "glue", # for string pasting
  "progressr" # live progress and ETA
)

github_packages <- c(
  "r-lib/styler" # Code formatting
)

# Installation commands (commented out, for reference)
invisible(lapply(
  required_packages, function(pkg) {
    if (!require(pkg, character.only = TRUE)) {
      install.packages(pkg)
    }
  }
))
invisible(lapply(
  github_packages, function(repo) {
    if (!require(basename(repo), character.only = TRUE)) {
      remotes::install_github(repo)
    }
  }
))

# Load packages (assumes they are already installed)
invisible(lapply(required_packages, library, character.only = TRUE))
invisible(lapply(basename(github_packages), library, character.only = TRUE))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc/drg-pipeline

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy


Loading required package: future

Loading required package: future.apply

Loading

# R Scripts


In [3]:
year <- 2018

# Source each file sequentially
for (file in list.files(
  here::here("data-cleaning/r_scripts_v2"),
  pattern = "\\.R$", full.names = TRUE
)) {
  invisible(source(file))
}

message(year)


ℹ 2025-02-17 11:28:42.249629 > Setting client.id from options(googleAuthR.client_id)

All directories exist.


Total Rows via cached object: 11777674

Utilizing 4 cores (8 threads)


2018



In [4]:
to_bq <- TRUE
bq_dataset <- "phic_claims"
gcp_proj <- "drg-pipeline"
to_sample <- FALSE
years <- 2018:2023
to_flush <- FALSE # Set this to TRUE to trigger directory deletion
to_write <- TRUE # Set to FALSE to skip upload step

# Define directories
pre_dir <- here::here("data-cleaning", "data", "chkpts", "chkpt_4_thai_master_input")
post_dir <- here::here("data-cleaning", "data", "chkpts", "chkpt_5_thai_output")
bwt_dir <- here::here("~/drg-pipeline/data-cleaning/data/chkpts/chkpt_11_bwt")

# Delete and recreate directories if to_flush is TRUE
if (to_flush) {
  if (dir.exists(pre_dir)) {
    unlink(pre_dir, recursive = TRUE, force = TRUE) # Delete pre directory
  }
  if (dir.exists(post_dir)) {
    unlink(post_dir, recursive = TRUE, force = TRUE) # Delete post directory
  }
  if (dir.exists(bwt_dir)) {
    unlink(bwt_dir, recursive = TRUE, force = TRUE) # Delete post directory
  }
}

# Ensure directories exist after deletion (or if to_flush is FALSE)
dir.create(here::here(pre_dir, "txt_raw"), recursive = TRUE, showWarnings = FALSE)
dir.create(here::here(post_dir, "txt_raw"), recursive = TRUE, showWarnings = FALSE)
dir.create(here::here(bwt_dir), recursive = TRUE, showWarnings = FALSE)


In [5]:
# Construct and execute gsutil commands
system(paste0(
  "gsutil -m cp -n 'gs://phic-claims-checkpoints/pre-tdrg/*.txt' ",
  shQuote(here::here(pre_dir, "txt_raw"))
))

system(paste0(
  "gsutil -m cp -n 'gs://phic-claims-checkpoints/post-tdrg/*.TXT' ",
  shQuote(here::here(post_dir, "txt_raw"))
))


In [6]:
# Function to process files and save as .rds
process_files <- function(directory, prefix, year) {
  rds_path <- here::here(directory, "txt_rds", paste0(prefix, year, ".rds"))
  if (file.exists(rds_path)) {
    return()
  } # Skip if .rds already exists

  files <- Sys.glob(here::here(
    directory, "txt_raw",
    paste0("*_", year, "_*.*")
  ))
  if (length(files) == 0) {
    return()
  } # Skip if no files found

  dir.create(here::here(directory, "txt_rds"),
    recursive = TRUE, showWarnings = FALSE
  )

  dt_list <- lapply(files, fread, colClasses = "character", na.strings = "--")
  saveRDS(rbindlist(dt_list, use.names = TRUE, fill = TRUE),
    rds_path,
    compress = FALSE
  )
}

# Parallel processing for pre & post datasets
tasks <- expand.grid(year = years, type = c("pre", "post"))

invisible(mclapply(seq_len(nrow(tasks)), function(i) {
  row <- tasks[i, ]
  process_files(
    get(paste0(row$type, "_dir")),
    paste0(row$type, "_dt_"), row$year
  )
}, mc.cores = detectCores() - 1))


In [7]:
Sys.setenv(BIGQUERY_TEST_PROJECT = "drg-pipeline")
billing <- bq_test_project()

# Paths
log_file <-
  normalizePath("~/drg-pipeline/data-cleaning/debug/cache/upload-bwt.log",
    mustWork = FALSE
  )
progress_file <-
  normalizePath("~/drg-pipeline/data-cleaning/debug/cache/progress.txt",
    mustWork = FALSE
  )
chkpt_dir <- here::here(
  "data-cleaning", "data", "chkpts",
  "chkpt_5_thai_output", "bq_rds"
)

# Init
dir.create(chkpt_dir, recursive = TRUE, showWarnings = FALSE)
write("", file = log_file)
write("0", file = progress_file)
start_time <- Sys.time()

# Logging function
message_parallel <- function(...) {
  system(sprintf(
    'echo "%s" >> %s',
    paste0(..., collapse = ""), shQuote(log_file)
  ))
}

# Get total row count for each year
get_total_row_count <- function(year) {
  sql <- sprintf("SELECT COUNT(*) AS row_count FROM `drg_claims.thai_%d`", year)
  tb <- tryCatch(bq_project_query(billing, sql), error = function(e) NULL)
  df <- if (!is.null(tb)) {
    tryCatch(bq_table_download(tb),
      error = function(e) NULL
    )
  } else {
    NULL
  }
  if (is.null(df) || nrow(df) == 0) 0 else as.integer(df$row_count)
}

# Compute total rows across all years
years <- 2018:2023
total_rows <- sum(sapply(years, get_total_row_count))

message_parallel(
  sprintf("Total expected rows across all years: %d", total_rows)
)

# Function to fetch data in batches
fetch_and_save_bq_combined <- function(year, batch_size = 100000) {
  table_name <- sprintf("drg_claims.thai_%d", year)
  rds_path <- here::here(chkpt_dir, sprintf("bq_dt_%d.rds", year))
  if (file.exists(rds_path)) {
    return()
  }

  offset <- 0L
  batch_number <- 1L
  all_data <- list()

  while (TRUE) {
    sql <- sprintf(
      "SELECT * FROM `%s` LIMIT %d OFFSET %d",
      table_name, batch_size, offset
    )

    # Fetch a batch
    df <- tryCatch(bq_table_download(bq_project_query(billing, sql)),
      error = function(e) NULL
    )
    if (is.null(df) || nrow(df) == 0) break # Stop if no more rows

    all_data[[batch_number]] <- as.data.table(df)
    num_rows <- nrow(df)

    # Update progress **incrementally**
    system(
      sprintf(
        "echo $(($(cat %s) + %d)) > %s",
        progress_file, num_rows, progress_file
      )
    )
    completed_rows <- as.integer(readLines(progress_file))
    progress_percent <- round((completed_rows / total_rows) * 100, 2)

    # ETA calculation
    elapsed_time <- as.numeric(difftime(Sys.time(), start_time, units = "secs"))
    avg_time_per_row <- ifelse(completed_rows > 0,
      elapsed_time / completed_rows, 0
    )
    eta_seconds <- max(0, (total_rows - completed_rows) * avg_time_per_row)
    eta_formatted <- sprintf(
      "%02d:%02d:%02d", floor(eta_seconds / 3600),
      floor((eta_seconds %% 3600) / 60), round(eta_seconds %% 60)
    )

    # Log status update after **each batch** instead of after full queries
    message_parallel(
      sprintf(
        "Year %d: Batch %d (%d rows). Progress: %.2f%%, ETA %s.",
        year, batch_number, num_rows, progress_percent, eta_formatted
      )
    )

    offset <- offset + batch_size
    batch_number <- batch_number + 1L
  }

  # Save final data
  if (length(all_data) > 0) {
    saveRDS(rbindlist(all_data, use.names = TRUE, fill = TRUE),
      rds_path,
      compress = FALSE
    )
  }
}

# Open log in VS Code automatically
system("code -r ~/drg-pipeline/data-cleaning/debug/cache/upload-bwt.log")

# Run in parallel
mclapply(years, \(year) fetch_and_save_bq_combined(as.integer(year)),
  mc.cores = min(length(years), detectCores() - 1)
)

message_parallel("All years processed!")


[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL

[[6]]
NULL

In [8]:
# Define input and output directories
input_dir <-
  "~/drg-pipeline/data-cleaning/data/chkpts/chkpt_4_thai_master_input/txt_rds"
output_dir <-
  "~/drg-pipeline/data-cleaning/data/chkpts/chkpt_11_bwt"

# Ensure output directory exists
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

# Define function to process each year's data
process_year <- function(year) {
  input_file <- file.path(input_dir, paste0("pre_dt_", year, ".rds"))
  output_file <- file.path(output_dir, paste0("bwt_", year, ".rds"))

  # Skip processing if output file already exists
  if (file.exists(output_file)) {
    message(sprintf("Skipping Year %d: RDS already exists.", year))
    return(NULL)
  }

  if (!file.exists(input_file)) {
    message(sprintf("Skipping missing file: %s", input_file))
    return(NULL)
  }

  dt <- readRDS(input_file) # Load RDS file
  if ("AdmWt" %in% names(dt)) {
    # Save extracted column
    saveRDS(dt[, .(AdmWt)], output_file, compress = FALSE)
    message(sprintf("Saved: %s", output_file))
  } else {
    message(sprintf("Warning: 'AdmWt' column not found in %s", input_file))
  }
}

# Define years and run in parallel
mclapply(years, process_year, mc.cores = detectCores() - 1)

message("Extraction complete!")


[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL

[[6]]
NULL

Extraction complete!



In [9]:
input_dir <- normalizePath("~/drg-pipeline/data-cleaning/data/chkpts/chkpt_11_bwt")

# Function to upload data to BigQuery
upload_to_bq <- function(year) {
  input_file <- here::here(input_dir, paste0("bwt_", year, ".rds"))

  # Skip if file doesn't exist
  if (!file.exists(input_file)) {
    message(sprintf("Skipping Year %d: No data found.", year))
    return(NULL)
  }

  if (to_bq) {
    result <- readRDS(input_file) # Load RDS file

    # Define correct BQ table name
    bq_table_name <- paste0("bwt_", year)

    # Get full BQ table reference
    bq_tbl_ref <- bq_table(gcp_proj, bq_dataset, bq_table_name)

    print(bq_tbl_ref)
    # Attempt to delete the table if it exists
    tryCatch(
      bq_table_delete(bq_tbl_ref),
      error = function(e) {
        if (grepl("Not found", e, ignore.case = TRUE)) {
          message(sprintf("Year %d: Table does not exist, skipping deletion.", year))
        } else {
          stop(e)
        }
      }
    )

    # Create the BQ table if it does not exist
    tryCatch(
      bq_table_create(
        bq_tbl_ref,
        fields = fromJSON(here("data-cleaning/r_scripts_v2", "bq_schema_thai_bwt.json"), simplifyDataFrame = FALSE)
      ),
      error = function(e) {
        if (grepl("already exists", e, ignore.case = TRUE)) {
          message(sprintf("Year %d: Table already exists. Skipping creation and upload.", year))
        } else {
          stop(e)
        }
      }
    )

    if (to_write) {
      chunk_size <- 250000
      num_chunks <- ceiling(nrow(result) / chunk_size)

      message(sprintf("Uploading Year %d in %d chunks...", year, num_chunks))

      for (i in seq_len(num_chunks)) {
        chunk <- result[((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)), ]

        # Upload chunk to BQ
        bq_table_upload(
          bq_tbl_ref,
          values = chunk,
          write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
        )

        message(sprintf("Year %d: Uploaded chunk %d/%d.", year, i, num_chunks))
      }
    }
  }
}

# Define years and run in parallel
mclapply(years, upload_to_bq,
  mc.cores = detectCores() - 1
)

message("All BQ uploads complete!")


[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL

[[6]]
NULL

All BQ uploads complete!

